# 01 — TimesFM 2.5 pipeline (univariate)

Base pipeline template using **TimesFM 2.5 200M** (Google) in zero-shot mode, without covariates.

**Environment:** main `.venv` (see `requirements.txt`).

**Expected data:** parquet with the schema described in [docs/01_dataset_overview.md](../docs/01_dataset_overview.md). Anonymized identifiers (`Inst_NN`, `Pub_X`).

**No executed outputs** — pedagogical template.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.data_loader import load_parquet, filter_period, build_series, input_matrix
from src.metrics import all_metrics, seasonal_naive

PARQUET_PATH = REPO_ROOT / 'data' / 'anonymized_series.parquet'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'timesfm_bare'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 12
YEAR_START = 2020
YEAR_END = 2024

## 2. Data loading

The standard function from `src/data_loader.py` is reused. The identifiers already come anonymized.

In [ ]:
df = load_parquet(PARQUET_PATH)
df = filter_period(df, YEAR_START, YEAR_END)
series = build_series(df, min_months=24)
print(f'Series loaded: {len(series)}')

train_list, test_list, ids = input_matrix(series, horizon=HORIZON)

## 3. Loading the TimesFM 2.5 model

Checkpoint: `google/timesfm-2.5-200m-pytorch`. Standard configuration described in [docs/02_models.md](../docs/02_models.md).

In [ ]:
from timesfm import TimesFM_2p5_200M_torch, ForecastConfig

model = TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
model.compile(ForecastConfig(
    max_context=1024,
    max_horizon=256,
    normalize_inputs=True,
    use_continuous_quantile_head=True,
    fix_quantile_crossing=True,
    return_backcast=False,
))

## 4. Inference

In [ ]:
point_forecasts, quantiles = model.forecast(
    horizon=HORIZON,
    inputs=train_list,
)
forecasts = np.clip(point_forecasts[:, :HORIZON], 0, None)

## 5. Evaluation with the 6 metrics

In [ ]:
rows = []
for i, sid in enumerate(ids):
    y_true = test_list[i]
    y_pred = forecasts[i]
    naive_pred = seasonal_naive(train_list[i], HORIZON)
    model_metrics = all_metrics(y_true, y_pred, train_list[i])
    naive_metrics = all_metrics(y_true, naive_pred, train_list[i])
    rows.append({'series_id': sid, 'model': 'TimesFM_bare', **model_metrics})
    rows.append({'series_id': sid, 'model': 'Seasonal_Naive', **naive_metrics})

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUTPUT_DIR / 'metrics_timesfm_bare.csv', index=False)
metrics_df.groupby('model')[['MASE', 'MAE', 'RMSE', 'sMAPE', 'MedAE']].median()

## 6. Detailed forecasts (for later comparisons)

In [ ]:
pred_rows = []
for i, sid in enumerate(ids):
    for h in range(HORIZON):
        pred_rows.append({
            'series_id': sid,
            'horizon_month': h + 1,
            'y_true': float(test_list[i][h]),
            'pred_timesfm_bare': float(forecasts[i, h]),
        })
pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(OUTPUT_DIR / 'predictions_timesfm_bare.csv', index=False)
pred_df.head()